# FunctionGemma-270M — fine-tune on Aura's tools

Complete rewrite of the official Unsloth `FunctionGemma_(270M)` notebook, wired to
**Aura's tool-calling dataset** (spike 073). Run top-to-bottom on a Colab **GPU**
runtime (T4 is plenty for 270M).

**What it does**
1. Installs Unsloth.
2. Loads the trainable base `unsloth/functiongemma-270m-it` (safetensors — NOT the GGUF).
3. Loads `train.jsonl` / `eval.jsonl` (you upload them) — HF `{messages, tools}` rows.
4. Formats with `apply_chat_template(messages, tools=...)`, masks loss to the model turn.
5. Fine-tunes (LoRA), smoke-tests, exports a **GGUF** you download for llama.cpp.

**Baseline to beat** (spike 071, base model, zero-shot on Aura tools): top-1 ≈ 8%,
~80% refusal. The goal is to lift Italian tool-firing + argument correctness.


## 1 · Install Unsloth

In [ ]:
%%capture
# Colab: installs unsloth + a compatible torch/trl/peft stack.
!pip install unsloth


## 2 · Load the base model

The **safetensors** base is the finetunable artifact; the `-GGUF` repo is inference-only.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 16384  # full 21-tool catalog per row measured ~13k tokens; 16k fits (Gemma-3 ctx = 32k)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/functiongemma-270m-it",
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = False,   # 270M in 16-bit is tiny; QLoRA unnecessary
    load_in_16bit  = True,
    full_finetuning= False,
)
print("loaded:", model.config._name_or_path)


## 3 · Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)


## 4 · Upload Aura's dataset

Run this cell, then pick `train.jsonl` and `eval.jsonl` (generated by
`go run ./.planning/spikes/073-fc270m-dataset-from-registry`).

Each row is `{"messages":[{user},{assistant w/ tool_calls}], "tools":[<catalog>]}`.


In [ ]:
from google.colab import files
up = files.upload()   # choose train.jsonl and eval.jsonl
print("uploaded:", list(up.keys()))


## 5 · Format rows → training text

`apply_chat_template` injects FunctionGemma's developer preamble + `<start_function_declaration>` block from `tools=`, then renders the model's tool-call turn.

In [ ]:
from datasets import load_dataset
import json

ds = load_dataset("json", data_files={"train": "train.jsonl", "eval": "eval.jsonl"})

def to_text(row):
    msgs = row["messages"]
    # Robustness: some template versions want tool_call arguments as a JSON string.
    # Aura emits them as objects; coerce to objects if a string slipped in.
    for m in msgs:
        for tc in (m.get("tool_calls") or []):
            args = tc["function"].get("arguments")
            if isinstance(args, str):
                try: tc["function"]["arguments"] = json.loads(args)
                except Exception: pass
    text = tokenizer.apply_chat_template(
        msgs, tools=row["tools"],
        add_generation_prompt=False, tokenize=False,
    )
    return {"text": text}

# Keep the full catalog for the inference cell BEFORE dropping columns.
CATALOG = ds["train"][0]["tools"]

# CRITICAL: drop messages/tools so TRL sees a pure-text dataset and does NOT take
# its conversational branch (which silently empties the split → num_samples=0).
train_ds = ds["train"].map(to_text, remove_columns=ds["train"].column_names)
eval_ds  = ds["eval"].map(to_text,  remove_columns=ds["eval"].column_names)
print("train", len(train_ds), "eval", len(eval_ds))
assert len(train_ds) > 0, "train split is empty — check the upload / JSONL paths"

# Length probe: every row carries the full 21-tool catalog, so confirm none exceed
# MAX_SEQ_LEN (rows longer than the trainer's max get filtered → could empty the split).
_lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in train_ds["text"]]
print(f"token length — max {max(_lens)}  mean {sum(_lens)//len(_lens)}  (MAX_SEQ_LEN={MAX_SEQ_LEN})")
assert max(_lens) <= MAX_SEQ_LEN, "some rows exceed MAX_SEQ_LEN — raise it or shrink the catalog"


### Sanity-check one formatted example

Confirm you see `developer` + `<start_function_declaration>` + the `<start_function_call>call:...` target before training.

In [ ]:
print(train_ds[0]["text"][:3000])


## 6 · Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 1,   # 16k-token rows are memory-heavy on a T4
        gradient_accumulation_steps = 8,   # effective batch 8
        warmup_steps                = 5,
        num_train_epochs            = 3,   # small dataset → a few epochs; watch eval loss
        learning_rate               = 2e-4,
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        report_to                   = "none",
        output_dir                  = "outputs",
    ),
)


## 7 · Mask loss to the model turn only

Gemma turn markers — train only on the assistant/tool-call response, not the prompt + declarations.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)


## 8 · Train

In [ ]:
stats = trainer.train()
print(stats)


## 9 · Smoke-test the finetuned model

The documented bitcoin gravity-well case + a couple more. Expect `web_search`, `mail__send_email`, etc. — in FunctionGemma's `<start_function_call>call:NAME{...}` format.

In [ ]:
FastLanguageModel.for_inference(model)
# CATALOG was captured in the format cell (before remove_columns dropped the tools column).

def ask(q):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tools=CATALOG, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=128,
                         do_sample=False, temperature=None, top_p=None, top_k=None)
    print("Q:", q)
    print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False))
    print("-" * 60)

for q in [
    "quanto costa il bitcoin adesso?",
    "manda una mail a Marco dicendo che arrivo tardi",
    "che ore sono?",
    "trova tutti i file con estensione go nel progetto",
]:
    ask(q)


## 10 · Export GGUF for llama.cpp

Unsloth merges the LoRA and converts to GGUF. `q8_0` keeps the 270M tiny (~290 MB) with near-bf16 quality.

In [ ]:
model.save_pretrained_gguf(
    "functiongemma-270m-aura-gguf",
    tokenizer,
    quantization_method = "q8_0",
)


## 11 · Download the GGUF

In [ ]:
import glob
from google.colab import files
ggufs = glob.glob("functiongemma-270m-aura-gguf/*.gguf")
print("artifacts:", ggufs)
files.download(ggufs[0])   # → serve locally with llama.cpp


## 12 · Serve + evaluate locally

```bash
docker run -d --name fc270m-aura --gpus all -p 127.0.0.1:8097:8097 \
  -v "$PWD:/models" ghcr.io/ggml-org/llama.cpp:server-cuda \
  -m /models/functiongemma-270m-aura-q8_0.gguf --jinja --host 0.0.0.0 --port 8097 --ctx-size 8192 -ngl 99
```

Then re-run the **spike-071 harness** against it for the head-to-head scorecard
(spike 074):

```bash
FC_BASE_URL=http://127.0.0.1:8097 FC_TAG=gpu-aura-ft go run ./.planning/spikes/071-fc270m-baseline-and-slot
FC_BASE_URL=http://127.0.0.1:8097 FC_TAG=aura-ft-EN FC_LANG=en go run ./.planning/spikes/071-fc270m-baseline-and-slot
```

Baseline to beat (071): top-1 ≈ 1/12, ~80% refusal. Finetuned target: high valid-call
rate + correct tool + correct args on the Italian set.

**Optional — push to Hugging Face instead of downloading:**
```python
model.push_to_hub_gguf("YOUR_HF_USER/functiongemma-270m-aura-GGUF",
                       tokenizer, quantization_method="q8_0", token="hf_...")
```
